In [1]:
import tensorflow as tf
import json 
import re
import numpy as np
from collections import defaultdict


2025-04-11 00:16:40.980531: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-11 00:16:40.983650: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-11 00:16:40.994254: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744330601.012613  130174 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744330601.018180  130174 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1744330601.031048  130174 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [2]:
with tf.io.gfile.GFile('gs://multi-robot-bucket2/data/generated_data/libero_90_reasonings.json') as f:
    reasonings = json.load(f)

reasonings['0'].keys()

dict_keys(['obj_id_to_name', 'language_label', 'end_effector_centroids', 'obj_masks'])

In [3]:
reasonings['0']['obj_id_to_name']

{'obj0': 'black book 1',
 'obj1': 'red coffee mug 1',
 'obj2': 'porcelain mug 1',
 'obj3': 'desk caddy 1'}

In [4]:
reasonings['0']['obj_masks']['obj0']

{'0': '<loc0507><loc0279><loc0741><loc0379>',
 '1': '<loc0507><loc0279><loc0741><loc0379>',
 '2': '<loc0507><loc0279><loc0741><loc0379>',
 '3': '<loc0507><loc0279><loc0741><loc0379>',
 '4': '<loc0507><loc0279><loc0741><loc0379>',
 '5': '<loc0507><loc0279><loc0741><loc0379>',
 '6': '<loc0507><loc0279><loc0741><loc0379>',
 '7': '<loc0507><loc0279><loc0741><loc0379>',
 '8': '<loc0507><loc0279><loc0741><loc0379>',
 '9': '<loc0507><loc0279><loc0741><loc0379>',
 '10': '<loc0507><loc0279><loc0741><loc0379>',
 '11': '<loc0507><loc0279><loc0741><loc0379>',
 '12': '<loc0507><loc0279><loc0741><loc0379>',
 '13': '<loc0507><loc0279><loc0741><loc0379>',
 '14': '<loc0507><loc0279><loc0741><loc0379>',
 '15': '<loc0507><loc0279><loc0741><loc0379>',
 '16': '<loc0507><loc0279><loc0741><loc0379>',
 '17': '<loc0507><loc0279><loc0741><loc0379>',
 '18': '<loc0507><loc0279><loc0741><loc0379>',
 '19': '<loc0507><loc0279><loc0741><loc0379>',
 '20': '<loc0507><loc0279><loc0741><loc0379>',
 '21': '<loc0507><loc02

In [15]:
# helpers
with tf.io.gfile.GFile('gs://multi-robot-bucket2/data/generated_data/libero_90_reasonings.json') as f:
    reasonings = json.load(f)
    
def remove_irrelevant_objs(obj_id_to_name, language_label):
    def is_relevant(obj_name, language_label):
        words_in_obj_name = obj_name.split(" ")
        
        # at least one word from the object name should be present in the language label
        if any([True if word in language_label else False for word in words_in_obj_name]):
            return True
        return False

    relevant_ids_to_name = {}
    for obj_id, name in obj_id_to_name.items():
        # heuristic 
        if is_relevant(name, language_label):
            relevant_ids_to_name[obj_id] = name
        else:
            print(f"removed {name}")

    
    return relevant_ids_to_name


def extract_bbox_coordinates(loc_string):
    """Extract the bounding box coordinates from the location string."""
    # Extract the numbers from the loc tags using regex
    if loc_string == "":
        return None
    
    pattern = r'<loc(\d+)>'
    matches = re.findall(pattern, loc_string)
    
    # Convert matches to integers
    if len(matches) != 4:
        raise ValueError(f"Expected 4 location tokens, got {len(matches)}")
    
    # Parse as [ymin, xmin, ymax, xmax]
    coords = [int(match) for match in matches]
    return coords

def calculate_bbox_movement(bbox_dict, min_point_movement=5):

    # Store coordinates for each frame
    frames = []
    
    # Sort keys numerically to ensure we process in order
    for key in sorted(bbox_dict.keys(), key=lambda x: int(x)):
        coords = extract_bbox_coordinates(bbox_dict[key])
        if coords is not None:
            frames.append(coords)
    
    if len(frames) <= 1:
        return {
            'total_distance': 0,
            'has_sufficient_movement': False,
            'corners_movement': [0, 0, 0, 0],
            'center_movement': 0
        }
    
    
    # Calculate center point of each bbox
    centers = []
    for i, frame in enumerate(frames):
        ymin, xmin, ymax, xmax = frame
        center_y = (ymin + ymax) / 2
        center_x = (xmin + xmax) / 2
        centers.append((center_x, center_y))
    centers = np.array(centers)
    
    # Calculate displacement between consecutive frames for the center
    center_displacements = np.sqrt(np.sum(np.diff(centers, axis=0)**2, axis=1))
    center_total_movement = np.sum(center_displacements)
    
    # Track movement of each corner of the bounding box
    # Corner order: top-left, top-right, bottom-left, bottom-right
    corners = np.zeros((len(frames), 4, 2))
    for i, frame in enumerate(frames):
        ymin, xmin, ymax, xmax = frame
        corners[i, 0] = [xmin, ymin]  # top-left
        corners[i, 1] = [xmax, ymin]  # top-right
        corners[i, 2] = [xmin, ymax]  # bottom-left
        corners[i, 3] = [xmax, ymax]  # bottom-right
    
    # Calculate total movement for each corner
    corner_movements = []
    for c in range(4):
        corner_track = corners[:, c, :]
        corner_displacements = np.sqrt(np.sum(np.diff(corner_track, axis=0)**2, axis=1))
        corner_total = np.sum(corner_displacements)
        corner_movements.append(corner_total)
    
    # Check if all corners have moved at least the minimum amount
    has_sufficient_movement = all(movement >= min_point_movement for movement in corner_movements)
    
    # Calculate cumulative distance traveled by center
    cumulative_distance = np.cumsum(center_displacements)
    total_distance = cumulative_distance[-1] if len(cumulative_distance) > 0 else 0
    
    return {
        'centers': centers,
        'displacements': center_displacements,
        'cumulative_distance': cumulative_distance,
        'total_distance': total_distance,
        'has_sufficient_movement': has_sufficient_movement,
        'corners_movement': corner_movements,
        'center_movement': center_total_movement
    }

def is_bbox_static(obj_trace, dist_thresh=100, point_thresh=10):
    results = calculate_bbox_movement(obj_trace, min_point_movement=point_thresh)
    
    # Check if total distance is below threshold or if corners haven't moved enough
    if results['total_distance'] < dist_thresh or not results['has_sufficient_movement']:
        return True
    return False

In [16]:
def compute_bbox_center(coords):
    ymin, xmin, ymax, xmax = coords
    return np.array([(ymin + ymax) / 2, (xmin + xmax) / 2])

def compute_bbox_center_and_area(coords):
    ymin, xmin, ymax, xmax = coords
    center = np.array([(ymin + ymax) / 2, (xmin + xmax) / 2])
    area = max(0, (ymax - ymin)) * max(0, (xmax - xmin))
    return center, area

def movement_score(bbox_trace, lambda_penalty=0.2):
    centers = []
    areas = []
    for frame_id in sorted(bbox_trace.keys(), key=lambda k: int(k)):
        coords = extract_bbox_coordinates(bbox_trace[frame_id])
        if coords is not None:
            center, area = compute_bbox_center_and_area(coords)
            centers.append(center)
            areas.append(area)

    if len(centers) < 2:
        return 0

    center_diffs = [np.linalg.norm(centers[i+1] - centers[i]) for i in range(len(centers) - 1)]
    area_diffs = [abs(areas[i+1] - areas[i]) for i in range(len(areas) - 1)]

    return sum(center_diffs) - lambda_penalty * sum(area_diffs)

def strip_suffix(name):
    return re.sub(r"\s*\d+$", "", name.strip())

def select_most_moved_objects(reasoning_dct, lambda_penalty=0.2):
    obj_id_to_name = reasoning_dct["obj_id_to_name"]
    obj_masks = reasoning_dct["obj_masks"]

    # Group obj_ids by base name
    base_name_to_obj_ids = defaultdict(list)
    for obj_id, name in obj_id_to_name.items():
        base_name = strip_suffix(name)
        base_name_to_obj_ids[base_name].append(obj_id)

    # Output dicts
    filtered_obj_id_to_name = {}
    filtered_obj_masks = {}

    for base_name, obj_ids in base_name_to_obj_ids.items():
        if len(obj_ids) == 1:
            # Only one object with this base name — keep it
            obj_id = obj_ids[0]
        else:
            # Choose object with most *real* movement
            best_score = -float("inf")
            best_id = None
            for obj_id in obj_ids:
                bbox_trace = obj_masks.get(obj_id, {})
                score = movement_score(bbox_trace, lambda_penalty=lambda_penalty)
                if score > best_score:
                    best_score = score
                    best_id = obj_id
            obj_id = best_id

        # Keep this obj_id
        filtered_obj_id_to_name[obj_id] = obj_id_to_name[obj_id]
        filtered_obj_masks[obj_id] = obj_masks[obj_id]

    return {
        "obj_id_to_name": filtered_obj_id_to_name,
        "obj_masks": filtered_obj_masks
    }


In [17]:
new_reasonings = {}
for traj_id, reasoning_dct in reasonings.items():
    print(f"PROCESSING TRAJECTORY {traj_id}: {reasoning_dct['language_label']} ************************************")
    lang_relevant_ids = remove_irrelevant_objs(
        reasoning_dct['obj_id_to_name'], 
        reasoning_dct['language_label']
    )
    lang_relevant_obj_masks = {id: reasoning_dct['obj_masks'][id] for id in lang_relevant_ids}

    reasoning_dct['obj_masks'] = lang_relevant_obj_masks
    reasoning_dct['obj_id_to_name'] = lang_relevant_ids

    # then, remove static objects
    non_static_obj_masks = {}
    non_static_relevant_ids = {}
    for id, obj_trace in reasoning_dct['obj_masks'].items():
        if not is_bbox_static(obj_trace):
            non_static_obj_masks[id] = obj_trace
            non_static_relevant_ids[id] = reasoning_dct['obj_id_to_name'][id]
        else:
            print(f"removed {reasoning_dct['obj_id_to_name'][id]}")


    reasoning_dct['obj_masks'] = lang_relevant_obj_masks
    reasoning_dct['obj_id_to_name'] = non_static_relevant_ids

    # lastly, find all objects with the same name, and keep only the one which moves the most
    final_ids_and_masks= select_most_moved_objects(reasoning_dct)
    reasoning_dct['obj_masks'] = final_ids_and_masks['obj_masks']
    reasoning_dct['obj_id_to_name'] = final_ids_and_masks['obj_id_to_name']

    new_reasonings[traj_id] = reasoning_dct
            

PROCESSING TRAJECTORY 0: pick up the book and place it in the left compartment of the caddy ************************************
removed red coffee mug 1
removed porcelain mug 1
removed desk caddy 1
PROCESSING TRAJECTORY 1: pick up the book on the right and place it on the cabinet shelf ************************************
removed black book 1
removed yellow book 2
removed wooden two layer shelf 1
PROCESSING TRAJECTORY 2: pick up the book on the right and place it on the cabinet shelf ************************************
removed black book 1
removed yellow book 2
removed wooden two layer shelf 1
PROCESSING TRAJECTORY 3: pick up the yellow and white mug and place it to the right of the caddy ************************************
removed black book 1
removed desk caddy 1
PROCESSING TRAJECTORY 4: open the top drawer of the cabinet and put the bowl in it ************************************
removed plate 1
removed wooden cabinet 1
PROCESSING TRAJECTORY 5: pick up the book and place it in th

PROCESSING TRAJECTORY 24: put the white bowl on the plate ************************************
removed microwave 1
removed plate 1
PROCESSING TRAJECTORY 25: put the red mug on the plate ************************************
removed chocolate pudding 1
removed plate 1
PROCESSING TRAJECTORY 26: put the wine bottle on the wine rack ************************************
removed akita black bowl 1
removed white cabinet 1
removed wine rack 1
PROCESSING TRAJECTORY 27: pick up the milk and put it in the basket ************************************
removed alphabet soup 1
removed cream cheese 1
removed tomato sauce 1
removed ketchup 1
removed orange juice 1
removed butter 1
removed basket 1
PROCESSING TRAJECTORY 28: open the top drawer of the cabinet ************************************
removed akita black bowl 1
removed plate 1
removed wooden cabinet 1
PROCESSING TRAJECTORY 29: pick up the book and place it in the left compartment of the caddy ************************************
removed white ye

In [18]:
with open('/nfs/nfs2/users/riadoshi/bigvision_palivla/data_generation/visualization/libero/libero_90_reasonings.json', 'w') as f:
    json.dump(new_reasonings, f)